# Vigil — Notebook 04: Análisis de Redes Sociales (SNA)

Este cuaderno implementa el análisis estructural del ecosistema: sincronía CIB (comportamiento inauténtico coordinado), agrupamiento de comunidades con Louvain y núcleo de red con K-Core.

In [ ]:
import polars as pl
import sys
sys.path.append('..')
from src.graph.network_analyzer import build_entity_network, detect_louvain_communities, calculate_k_core, calcular_sincronia_cib

## 1. Cargar Datos Enriquecidos (Capa Gold)

In [ ]:
posts_gold = pl.read_parquet("../data/gold/redes_sociales/fb_posts_gold.parquet")
comments_gold = pl.read_parquet("../data/gold/redes_sociales/fb_comments_gold.parquet")

## 2. Detección CIB por Sincronía Temporal ($S_T$)

In [ ]:
alertas_cib = calcular_sincronia_cib(posts_gold, delta_t_max=300.0)

print(f"Se encontraron {len(alertas_cib)} alertas de coordinación sospechosa.")
for idx, alerta in enumerate(alertas_cib):
    print(f"Alerta {idx+1}:")
    print(f"  Páginas: {alerta['paginas_involucradas']}")
    print(f"  Sincronía S_T: {alerta['sincronia_s_t']}")
    print(f"  Texto: {alerta['texto_coordinado']}")

## 3. Construcción del Grafo Semántico de Co-ocurrencia

In [ ]:
# Unificar datasets para extraer co-ocurrencia de keywords/entidades
df_combined = pl.concat([
    posts_gold.select(["keywords_extracted", "texto_publicacion"]),
    comments_gold.select(["keywords_extracted", "texto_publicacion"])
])

G = build_entity_network(df_combined)

## 4. Detección de Comunidades (Louvain)

In [ ]:
partition = detect_louvain_communities(G)

# Agrupar nodos por ID de comunidad
comunidades_dict = {}
for nodo, com_id in partition.items():
    comunidades_dict.setdefault(com_id, []).append(nodo)

print("Distribución de Comunidades Temáticas:")
for com_id, nodos in comunidades_dict.items():
    print(f"Comunidad {com_id} (Nodos: {len(nodos)}): {nodos[:10]}...")

## 5. Extracción del Núcleo de Red (K-Core)

In [ ]:
G_core = calculate_k_core(G, k=2)

print(f"Nodos nucleares identificados (grado >= 2): {len(G_core.nodes)}")
print("Entidades del Núcleo duro:", list(G_core.nodes))